In [45]:
from pathlib import Path
from typing import Any

import optuna
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor

In [46]:
Path.cwd()

PosixPath('/Users/cube/source/dhbw/exploration/src2')

In [47]:
root_dir = Path.cwd().parent
temp_dir = root_dir / ".temp"
assert temp_dir.exists()

temp_dir

PosixPath('/Users/cube/source/dhbw/exploration/.temp')

In [48]:
dataset_train_path = temp_dir / "vehicles_train.csv"
dataset_test_path = temp_dir / "vehicles_test.csv"
dataset_train_path, dataset_test_path

(PosixPath('/Users/cube/source/dhbw/exploration/.temp/vehicles_train.csv'),
 PosixPath('/Users/cube/source/dhbw/exploration/.temp/vehicles_test.csv'))

In [49]:
images_path = Path.cwd() / ".." / "charged-ieee" / "images"
images_path

PosixPath('/Users/cube/source/dhbw/exploration/src2/../charged-ieee/images')

Globals

In [50]:
RNG = 99

# Load

In [51]:
df_train, df_test = pd.read_csv(dataset_train_path), pd.read_csv(dataset_test_path)
df_train.shape, df_test.shape

((279106, 15), (93036, 15))

In [52]:
cols_iso = ["year", "odometer", "manufacturer_missing"]

# Prepare

In [53]:
col_label = "price"
cols_features = [c for c in df_train.columns if c != col_label]
col_label, len(cols_features)

('price', 14)

In [54]:
X_train, y_train = df_train[cols_features], df_train[col_label]
X_test, y_test = df_test[cols_features], df_test[col_label]
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((279106, 14), (279106,), (93036, 14), (93036,))

# Train

### Common

In [55]:
def suggest_isolation_forest_params(trial: optuna.Trial) -> dict[str, Any]:
    mode = trial.suggest_categorical("iso_contamination_mode", ["auto", "manual"])

    return {
        "iso_n_estimators": trial.suggest_int("iso_n_estimators", 50, 200),
        "iso_contamination": "auto"
        if mode == "auto"
        else trial.suggest_float("iso_contamination", 0.01, 0.5),
        "iso_contamination_mode": mode,
    }

In [56]:
def new_isolation_forest(model_params, **params) -> IsolationForest:
    return IsolationForest(
        n_estimators=model_params["iso_n_estimators"],
        contamination="auto"
        if model_params["iso_contamination_mode"] == "auto"
        else model_params["iso_contamination"],
        random_state=RNG,
        **params,
    )


def fit_model_with_isolation_forest(model, X_in, y_in, model_params):
    model_iso = new_isolation_forest(model_params, n_jobs=-1)
    inlier_mask = model_iso.fit_predict(X_in[cols_iso]) == 1
    model.fit(X_in[inlier_mask], y_in[inlier_mask])
    return model

In [57]:
def new_model_pipeline(model) -> Pipeline:
    cols_cat_target = ["region", "state", "manufacturer"]
    cols_cat_one_hot = [
        "condition",
        "cylinders",
        "fuel",
        "title_status",
        "transmission",
        "drive",
        "type",
        "paint_color",
    ]
    cols_num = ["year", "odometer", "manufacturer_missing"]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "target",
                TargetEncoder(target_type="continuous", random_state=RNG),
                cols_cat_target,
            ),
            (
                "one_hot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                cols_cat_one_hot,
            ),
            (
                "num",
                "passthrough",
                cols_num,
            ),
        ],
        sparse_threshold=0,
        verbose_feature_names_out=False,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("scaler", StandardScaler()),
            ("model", model),
        ]
    )

### Isolation Forest Parameters

In [58]:
isolation_forest_params = {
    "iso_n_estimators": 157,
    "iso_contamination_mode": "manual",
    "iso_contamination": 0.4662846821308693,
}
isolation_forest_params

{'iso_n_estimators': 157,
 'iso_contamination_mode': 'manual',
 'iso_contamination': 0.4662846821308693}

### Model Dummy (Median)

In [59]:
model_dummy = new_model_pipeline(DummyRegressor(strategy="median"))
model_dummy.fit(X_train, y_train)
y_pred_dummy = model_dummy.predict(X_test)
mae_dummy, rmse_dummy, r2_dummy = (
    mean_absolute_error(y_test, y_pred_dummy),
    root_mean_squared_error(y_test, y_pred_dummy),
    r2_score(y_test, y_pred_dummy),
)
mae_dummy, rmse_dummy, r2_dummy

(10972.87008254869, 14264.327880967614, -0.04763379647935784)

### Model Linear

In [60]:
model_linear = new_model_pipeline(LinearRegression(n_jobs=-1))
model_linear.fit(X_train, y_train)
y_pred_linear = model_linear.predict(X_test)
mae_linear, rmse_linear, r2_linear = (
    mean_absolute_error(y_test, y_pred_linear),
    root_mean_squared_error(y_test, y_pred_linear),
    r2_score(y_test, y_pred_linear),
)
mae_linear, rmse_linear, r2_linear

(6142.9573196718175, 8707.342244014852, 0.6096284996005732)

### Model RandomForest

In [61]:
def new_model_rft(model_params, **params) -> RandomForestRegressor:
    return RandomForestRegressor(
        n_estimators=model_params["rf_n_estimators"],
        max_depth=model_params["rf_max_depth"],
        min_samples_split=model_params["rf_min_samples_split"],
        min_samples_leaf=model_params["rf_min_samples_leaf"],
        random_state=RNG,
        **params,
    )

In [62]:
def objective_model_rft(trial):
    iso = new_isolation_forest(
        suggest_isolation_forest_params(trial),
        n_jobs=-1,
    )
    inlier_mask = iso.fit_predict(X_train[cols_iso]) == 1
    X_in, y_in = X_train[inlier_mask], y_train[inlier_mask]

    model_pipeline = new_model_pipeline(
        new_model_rft(
            {
                "rf_n_estimators": trial.suggest_int("rf_n_estimators", 100, 500),
                "rf_max_depth": trial.suggest_int("rf_max_depth", 4, 24),
                "rf_min_samples_split": trial.suggest_int(
                    "rf_min_samples_split", 2, 20
                ),
                "rf_min_samples_leaf": trial.suggest_int("rf_min_samples_leaf", 1, 10),
            },
            n_jobs=-1,
        )
    )

    cv = KFold(n_splits=7, shuffle=True, random_state=RNG)
    mae = -cross_val_score(
        model_pipeline, X_in, y_in, cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1
    ).mean()
    return mae

In [ ]:
study_rft = optuna.create_study(direction="minimize")
study_rft.optimize(objective_model_rft, n_trials=25)
model_rft_params = study_rft.best_params
model_rft_params

[I 2026-04-20 14:43:22,625] A new study created in memory with name: no-name-31492bde-8186-4ccd-bb1f-31cd538cdb56
[I 2026-04-20 14:48:23,259] Trial 0 finished with value: 2853.7290727979207 and parameters: {'iso_contamination_mode': 'auto', 'iso_n_estimators': 165, 'rf_n_estimators': 328, 'rf_max_depth': 21, 'rf_min_samples_split': 16, 'rf_min_samples_leaf': 6}. Best is trial 0 with value: 2853.7290727979207.
[I 2026-04-20 14:53:23,715] Trial 1 finished with value: 2984.623367210032 and parameters: {'iso_contamination_mode': 'manual', 'iso_n_estimators': 102, 'iso_contamination': 0.14854935991628507, 'rf_n_estimators': 342, 'rf_max_depth': 21, 'rf_min_samples_split': 12, 'rf_min_samples_leaf': 10}. Best is trial 0 with value: 2853.7290727979207.
[I 2026-04-20 14:56:39,817] Trial 2 finished with value: 2597.3951647938075 and parameters: {'iso_contamination_mode': 'auto', 'iso_n_estimators': 144, 'rf_n_estimators': 214, 'rf_max_depth': 24, 'rf_min_samples_split': 10, 'rf_min_samples_leaf

In [ ]:
model_rft_params = {
    "iso_n_estimators": 86,
    "iso_contamination_mode": "auto",
    "rf_n_estimators": 418,
    "rf_max_depth": 23,
    "rf_min_samples_split": 8,
    "rf_min_samples_leaf": 4,
}

In [ ]:
model_rft = new_model_pipeline(new_model_rft(model_rft_params, n_jobs=-1))
model_rft = fit_model_with_isolation_forest(
    model_rft, X_train, y_train, model_rft_params
)
y_pred_rft = model_rft.predict(X_test)
mae_rft, rmse_rft, r2_rft = (
    mean_absolute_error(y_test, y_pred_rft),
    root_mean_squared_error(y_test, y_pred_rft),
    r2_score(y_test, y_pred_rft),
)
mae_rft, rmse_rft, r2_rft

(2985.251520593234, 5453.66379146577, 0.8468617795166383)

In [ ]:
model_rft2 = new_model_pipeline(new_model_rft(model_rft_params, n_jobs=-1))
model_rft2.fit(X_train, y_train)
y_pred_rft2 = model_rft2.predict(X_test)
mae_rft2, rmse_rft2, r2_rft2 = (
    mean_absolute_error(y_test, y_pred_rft2),
    root_mean_squared_error(y_test, y_pred_rft2),
    r2_score(y_test, y_pred_rft2),
)
mae_rft2, rmse_rft2, r2_rft2

(2533.760387278712, 4688.4508363206105, 0.8868210639923669)